In [2]:
import cv2 as cv
import numpy as np
import time
from concurrent.futures import ThreadPoolExecutor

def calcular_promedio(fragmento):
    """Calcula la media de intensidad de un cuadrante."""
    return float(np.mean(fragmento))

def obtener_cuadrantes(frame):
    """
    Retorna la lista de los 8 fragmentos y sus respectivas coordenadas (x1, y1, x2, y2)
    para poder dibujar rectángulos sobre ellos si cambian.
    """
    alto, ancho = frame.shape[:2]
    h_m = alto // 2
    w_c = ancho // 4

    coordenadas = [
        # Fila superior
        (0, 0, w_c, h_m),
        (w_c, 0, w_c * 2, h_m),
        (w_c * 2, 0, w_c * 3, h_m),
        (w_c * 3, 0, ancho, h_m),
        # Fila inferior
        (0, h_m, w_c, alto),
        (w_c, h_m, w_c * 2, alto),
        (w_c * 2, h_m, w_c * 3, alto),
        (w_c * 3, h_m, ancho, alto)
    ]

    fragmentos = [frame[y1:y2, x1:x2] for (x1, y1, x2, y2) in coordenadas]
    return fragmentos, coordenadas

# Inicializar cámara
camara = cv.VideoCapture(0, cv.CAP_V4L2)
if not camara.isOpened():
    camara = cv.VideoCapture(1, cv.CAP_V4L2)

if not camara.isOpened():
    print("Error: No se pudo abrir la cámara.")
    exit()

print("Cámara lista. Presiona 'q' en la ventana para salir.")

# Umbral de sensibilidad: cambios de intensidad mayores a este valor activan el recuadro
UMBRAL_CAMBIO = 5.0  

medias_anteriores = None

with ThreadPoolExecutor(max_workers=8) as executor:
    tiempo_ultima_actualizacion = time.time()
    INTERVALO_SEGUNDOS = 1.5  # Renovar referencia cada segundo
    while True:
        ret, frame = camara.read()
        if not ret:
            break

        # 1. Obtener fragmentos y sus posiciones geométricas
        fragmentos, coords = obtener_cuadrantes(frame)

        # 2. Calcular las 8 medias en paralelo
        medias_actuales = np.array(list(executor.map(calcular_promedio, fragmentos)))

        # 3. Comparar con el frame anterior
        if medias_anteriores is not None:
            # Diferencia absoluta por cuadrante
            diferencias = np.abs(medias_actuales - medias_anteriores)

            for i, dif in enumerate(diferencias):
                x1, y1, x2, y2 = coords[i]

                # Dibujar cuadrícula tenue de referencia (color gris)
                cv.rectangle(frame, (x1, y1), (x2, y2), (60, 60, 60), 1)

                # Si la diferencia supera el umbral, dibujar recuadro de alerta
                if dif > UMBRAL_CAMBIO:
                    # Rectángulo verde/rojo llamativo (BGR: verde brillante = (0, 255, 0))
                    cv.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
                    
                    # Mostrar el valor de la diferencia detectada sobre el cuadrante
                    texto = f"Dif: {dif:.1f}"
                    cv.putText(frame, texto, (x1 + 10, y1 + 30), 
                               cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

        # Actualizar medias anteriores para el próximo ciclo
        # (Opcional: usar un suavizado exponencial si quieres que se adapte lento a la luz:
        # medias_anteriores = 0.8 * medias_anteriores + 0.2 * medias_actuales)
        if (time.time() - tiempo_ultima_actualizacion) >= INTERVALO_SEGUNDOS:
            medias_anteriores = medias_actuales
            tiempo_ultima_actualizacion = time.time()
        cv.imshow('Deteccion de Cambios por Cuadrante', frame)

        if cv.waitKey(1) & 0xFF == ord('q'):
            break

time.sleep(0.1)
camara.release()
cv.destroyAllWindows()

Cámara lista. Presiona 'q' en la ventana para salir.
